In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuración Inicial ---
N_PROYECTOS = 4200
START_DATE = datetime(2019, 1, 1)
END_DATE = datetime(2025, 12, 31)
DIAS_RANGO_TOTAL = (END_DATE - START_DATE).days

print(f"Generando {N_PROYECTOS} proyectos simulados entre 2019 y 2025...")

# --- 1. Generar Códigos Únicos y Variables Categóricas (con sesgos realistas) ---

# Generar 4200 números únicos entre 1000 y 9999
codigos_numericos = np.random.choice(range(1000, 10000), N_PROYECTOS, replace=False)
codigos_proyecto = [f"SDATOOL-{num}" for num in codigos_numericos]

areas = ['CCS', 'ENG', 'RIC', 'RCS', 'FIN', 'T&C', 'GRM', 'CIB']
prioridades = ['Alta', 'Media', 'Baja']
complejidades = ['Alta', 'Media', 'Baja']

data = {
    'Código de proyecto': codigos_proyecto,
    
    # Asumimos que algunas áreas tienen más proyectos (ENG, FIN)
    'Área': np.random.choice(areas, N_PROYECTOS, p=[0.1, 0.2, 0.1, 0.1, 0.2, 0.1, 0.1, 0.1]),
    
    # Asumimos más proyectos de prioridad Media
    'Prioridad': np.random.choice(prioridades, N_PROYECTOS, p=[0.3, 0.5, 0.2]),
    
    'Complejidad técnica': np.random.choice(complejidades, N_PROYECTOS, p=[0.3, 0.5, 0.2]),
    
    'Número de dependencias asociadas': np.random.randint(1, 11, N_PROYECTOS),
    
    # Asumimos que la mayoría de proyectos se planean para 2 o 3 Q's
    'Time to market teórico en Q\'s': np.random.choice([1, 2, 3, 4], N_PROYECTOS, p=[0.15, 0.4, 0.35, 0.1]),
}

df = pd.DataFrame(data)

# --- 2. Generar Variables Correlacionadas (Presupuesto) ---

# El presupuesto DEBE estar correlacionado con la complejidad, dependencias y TTM teórico.
def calcular_presupuesto(row):
    base_presupuesto = 50000
    
    # Factor por complejidad
    if row['Complejidad técnica'] == 'Alta':
        factor_complejidad = 2.5
    elif row['Complejidad técnica'] == 'Media':
        factor_complejidad = 1.5
    else:
        factor_complejidad = 1.0
        
    # Factor por dependencias (más dependencias, más costo de gestión)
    factor_dependencias = 1 + (row['Número de dependencias asociadas'] / 10)
    
    # Factor por TTM (proyectos más largos cuestan más)
    factor_ttm = row['Time to market teórico en Q\'s'] * 0.5
    
    # Ruido aleatorio para que no sea una fórmula perfecta
    ruido = np.random.randint(-15000, 15000)
    
    presupuesto = (base_presupuesto * factor_complejidad * factor_dependencias * factor_ttm) + ruido
    return max(10000, int(presupuesto)) # Aseguramos un mínimo

df['Presupuesto'] = df.apply(calcular_presupuesto, axis=1)

# --- 3. Generar Fechas y Simulación de Retraso (El núcleo del modelo) ---

# Generar fechas de inicio aleatorias en el rango 2019-2025
# >>> ESTA ES LA LÍNEA CORREGIDA (VERSIÓN VECTORIZADA) <<<
dias_random = np.random.randint(0, DIAS_RANGO_TOTAL, N_PROYECTOS)
df['Fecha de inicio de ejecución'] = START_DATE + pd.to_timedelta(dias_random, unit='d')

# Convertir TTM Teórico de Q's a días (1 Q ≈ 91.25 días)
df['ttm_teorico_dias'] = df['Time to market teórico en Q\'s'] * 91.25

# --- Aquí está la "magia" para el modelo predictivo ---
# Vamos a simular el retraso (TTM Real - TTM Teórico) basado en las variables de entrada

def calcular_retraso_simulado(row):
    retraso_dias = 0
    
    # 1. La complejidad impacta el retraso
    if row['Complejidad técnica'] == 'Alta':
        retraso_dias += np.random.randint(30, 90) # Retraso alto
    elif row['Complejidad técnica'] == 'Media':
        retraso_dias += np.random.randint(0, 45)
        
    # 2. Las dependencias impactan MUCHO
    if row['Número de dependencias asociadas'] > 5:
        retraso_dias += row['Número de dependencias asociadas'] * np.random.randint(5, 15)
        
    # 3. El "Proyecto Apurado" (Alta Prioridad + TTM Teórico bajo) se retrasa
    if row['Prioridad'] == 'Alta' and row['Time to market teórico en Q\'s'] <= 2:
        retraso_dias += np.random.randint(20, 60)
        
    # 4. Impacto del Área
    if row['Área'] in ['ENG', 'CIB']:
        retraso_dias += np.random.randint(0, 30)
        
    # 5. Ruido aleatorio (algunos proyectos se adelantan)
    retraso_dias += np.random.randint(-20, 20)
    
    # Asegurarse de que el retraso no sea absurdamente negativo
    return max(-30, int(retraso_dias)) # Un proyecto se puede adelantar máx 30 días

# Aplicamos la simulación
df['retraso_simulado_dias'] = df.apply(calcular_retraso_simulado, axis=1)
df['ttm_real_dias'] = df['ttm_teorico_dias'] + df['retraso_simulado_dias']

# --- 4. Calcular Fechas Finales (hacia atrás desde la fecha final) ---

# Fecha final: Inicio + TTM Real
df['Fecha de uso por el personal'] = df['Fecha de inicio de ejecución'] + pd.to_timedelta(df['ttm_real_dias'], unit='d')

# Fecha pase a producción: Unos días ANTES de que lo use el personal
df['Fecha pase a producción'] = df['Fecha de uso por el personal'] - pd.to_timedelta(np.random.randint(1, 11, N_PROYECTOS), unit='d')

# Fecha fin de ejecución: Unos días/semanas ANTES del pase a producción (fase de pruebas/UAT)
df['Fecha fin de ejecución'] = df['Fecha pase a producción'] - pd.to_timedelta(np.random.randint(7, 30, N_PROYECTOS), unit='d')

# --- 5. Calcular TTM final y Limpieza ---

# Calcular Time to Market real en días
delta_ttm = (df['Fecha de uso por el personal'] - df['Fecha de inicio de ejecución'])
df['Time to market'] = delta_ttm.dt.days

# Formatear fechas a XX/XX/XXXX
df['Fecha de inicio de ejecución'] = df['Fecha de inicio de ejecución'].dt.strftime('%d/%m/%Y')
df['Fecha fin de ejecución'] = df['Fecha fin de ejecución'].dt.strftime('%d/%m/%Y')
df['Fecha pase a producción'] = df['Fecha pase a producción'].dt.strftime('%d/%m/%Y')
df['Fecha de uso por el personal'] = df['Fecha de uso por el personal'].dt.strftime('%d/%m/%Y')

# Definir el orden final de las columnas
columnas_finales = [
    'Código de proyecto',
    'Área',
    'Prioridad',
    'Presupuesto',
    'Complejidad técnica',
    'Número de dependencias asociadas',
    'Time to market teórico en Q\'s',
    'Fecha de inicio de ejecución',
    'Fecha fin de ejecución',
    'Fecha pase a producción',
    'Fecha de uso por el personal',
    'Time to market' # (Fecha uso personal - Fecha inicio)
]

df_final = df[columnas_finales]

# --- 6. Exportar ---
df_final.to_csv('proyectos_modelo_predictivo.csv', index=False, encoding='utf-8-sig', sep=';')
df_final.to_excel('proyectos_modelo_predictivo.xlsx', index=False)

print("\n--- ¡Simulación completada con éxito! ---")
print(f"Archivos guardados: 'proyectos_modelo_predictivo.csv' y 'proyectos_modelo_predictivo.xlsx'")
print("\n--- Vista previa de los primeros 5 proyectos generados ---")
print(df_final.head().to_string())

print("\n--- Vista previa del retraso simulado (Lo que tu modelo debe predecir) ---")
print(df[['ttm_teorico_dias', 'retraso_simulado_dias', 'ttm_real_dias', 'Time to market']].head())

Generando 4200 proyectos simulados entre 2019 y 2025...

--- ¡Simulación completada con éxito! ---
Archivos guardados: 'proyectos_modelo_predictivo.csv' y 'proyectos_modelo_predictivo.xlsx'

--- Vista previa de los primeros 5 proyectos generados ---
  Código de proyecto Área Prioridad  Presupuesto Complejidad técnica  Número de dependencias asociadas  Time to market teórico en Q's Fecha de inicio de ejecución Fecha fin de ejecución Fecha pase a producción Fecha de uso por el personal  Time to market
0       SDATOOL-4337  RCS      Alta       155411               Media                                 5                              3                   04/02/2022             04/10/2022              22/10/2022                   30/10/2022             268
1       SDATOOL-6376  FIN      Baja       216742                Alta                                 8                              2                   24/08/2025             21/07/2026              12/08/2026                   13/08/2026  

In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuración Inicial ---
N_PROYECTOS = 4200
# El inicio absoluto es el 1 de enero de 2019
ABSOLUTE_START_DATE = datetime(2019, 1, 1)
# El fin absoluto es el 30 de septiembre de 2025
ABSOLUTE_END_DATE = datetime(2025, 9, 30)

print(f"Generando {N_PROYECTOS} proyectos simulados entre 2019-01-01 y 2025-09-30...")

# --- 1. Generar Códigos Únicos y Variables Categóricas (con sesgos realistas) ---
# (Esta sección no cambia)

# Generar 4200 números únicos entre 1000 y 9999
codigos_numericos = np.random.choice(range(1000, 10000), N_PROYECTOS, replace=False)
codigos_proyecto = [f"SDATOOL-{num}" for num in codigos_numericos]

areas = ['CCS', 'ENG', 'RIC', 'RCS', 'FIN', 'T&C', 'GRM', 'CIB']
prioridades = ['Alta', 'Media', 'Baja']
complejidades = ['Alta', 'Media', 'Baja']

data = {
    'Código de proyecto': codigos_proyecto,
    'Área': np.random.choice(areas, N_PROYECTOS, p=[0.1, 0.2, 0.1, 0.1, 0.2, 0.1, 0.1, 0.1]),
    'Prioridad': np.random.choice(prioridades, N_PROYECTOS, p=[0.3, 0.5, 0.2]),
    'Complejidad técnica': np.random.choice(complejidades, N_PROYECTOS, p=[0.3, 0.5, 0.2]),
    'Número de dependencias asociadas': np.random.randint(1, 11, N_PROYECTOS),
    'Time to market teórico en Q\'s': np.random.choice([1, 2, 3, 4], N_PROYECTOS, p=[0.15, 0.4, 0.35, 0.1]),
}

df = pd.DataFrame(data)

# --- 2. Generar Variables Correlacionadas (Presupuesto) ---
# (Esta sección no cambia)

def calcular_presupuesto(row):
    base_presupuesto = 50000
    if row['Complejidad técnica'] == 'Alta': factor_complejidad = 2.5
    elif row['Complejidad técnica'] == 'Media': factor_complejidad = 1.5
    else: factor_complejidad = 1.0
    factor_dependencias = 1 + (row['Número de dependencias asociadas'] / 10)
    factor_ttm = row['Time to market teórico en Q\'s'] * 0.5
    ruido = np.random.randint(-15000, 15000)
    presupuesto = (base_presupuesto * factor_complejidad * factor_dependencias * factor_ttm) + ruido
    return max(10000, int(presupuesto))

df['Presupuesto'] = df.apply(calcular_presupuesto, axis=1)

# --- 3. Simulación de Retraso (Independiente de las fechas) ---
# (Esta sección calcula el TTM, pero aún no asigna fechas)

df['ttm_teorico_dias'] = df['Time to market teórico en Q\'s'] * 91.25

def calcular_retraso_simulado(row):
    retraso_dias = 0
    if row['Complejidad técnica'] == 'Alta': retraso_dias += np.random.randint(30, 90)
    elif row['Complejidad técnica'] == 'Media': retraso_dias += np.random.randint(0, 45)
    if row['Número de dependencias asociadas'] > 5: retraso_dias += row['Número de dependencias asociadas'] * np.random.randint(5, 15)
    if row['Prioridad'] == 'Alta' and row['Time to market teórico en Q\'s'] <= 2: retraso_dias += np.random.randint(20, 60)
    if row['Área'] in ['ENG', 'CIB']: retraso_dias += np.random.randint(0, 30)
    retraso_dias += np.random.randint(-20, 20)
    return max(-30, int(retraso_dias))

df['retraso_simulado_dias'] = df.apply(calcular_retraso_simulado, axis=1)
df['ttm_real_dias'] = df['ttm_teorico_dias'] + df['retraso_simulado_dias']
# Asegurarnos que el TTM real sea al menos positivo (ej. 30 días)
df['ttm_real_dias'] = df['ttm_real_dias'].clip(lower=30) 


# --- 4. Calcular Fechas Finales (LÓGICA INVERTIDA) ---
# ¡¡AQUÍ ESTÁ EL CAMBIO IMPORTANTE!!

# 4.1. Generar 'Fecha de uso por el personal' PRIMERO, asegurando que sea <= 2025-09-30
# El rango de fechas de finalización debe ser desde (Inicio 2019 + TTM Mínimo) hasta 2025-09-30
MIN_END_DATE = ABSOLUTE_START_DATE + timedelta(days=30) # Un proyecto dura al menos 30 días
DIAS_RANGO_FINALIZACION = (ABSOLUTE_END_DATE - MIN_END_DATE).days

dias_random_final = np.random.randint(0, DIAS_RANGO_FINALIZACION, N_PROYECTOS)
df['Fecha de uso por el personal'] = MIN_END_DATE + pd.to_timedelta(dias_random_final, unit='d')

# 4.2. Calcular 'Fecha de inicio' HACIA ATRÁS, restando el TTM real
df['Fecha de inicio de ejecución'] = df['Fecha de uso por el personal'] - pd.to_timedelta(df['ttm_real_dias'], unit='d')

# 4.3. APLICAR LÍMITE 2019: Anclar las fechas de inicio que sean < 2019-01-01
df['Fecha de inicio de ejecución'] = df['Fecha de inicio de ejecución'].clip(lower=ABSOLUTE_START_DATE)

# 4.4. Generar fechas intermedias (que ahora están garantizadas de estar en el rango)
# Fecha pase a producción: Unos días ANTES de que lo use el personal
df['Fecha pase a producción'] = df['Fecha de uso por el personal'] - pd.to_timedelta(np.random.randint(1, 11, N_PROYECTOS), unit='d')

# Fecha fin de ejecución: Unos días/semanas ANTES del pase a producción
df['Fecha fin de ejecución'] = df['Fecha pase a producción'] - pd.to_timedelta(np.random.randint(7, 30, N_PROYECTOS), unit='d')

# 4.5. Asegurarse que Fin Ejecución y Prod no sean anteriores a la Fecha de Inicio (por si el TTM es muy corto)
df['Fecha pase a producción'] = df['Fecha pase a producción'].clip(lower=df['Fecha de inicio de ejecución'])
df['Fecha fin de ejecución'] = df['Fecha fin de ejecución'].clip(lower=df['Fecha de inicio de ejecución'])


# --- 5. Calcular TTM final y Limpieza ---

# RE-CALCULAR el 'Time to market' basado en las fechas finales (que podrían estar "ancladas")
# Esto es vital para que el modelo tenga los datos correctos
delta_ttm = (df['Fecha de uso por el personal'] - df['Fecha de inicio de ejecución'])
df['Time to market'] = delta_ttm.dt.days

# Formatear fechas a XX/XX/XXXX
df['Fecha de inicio de ejecución'] = df['Fecha de inicio de ejecución'].dt.strftime('%d/%m/%Y')
df['Fecha fin de ejecución'] = df['Fecha fin de ejecución'].dt.strftime('%d/%m/%Y')
df['Fecha pase a producción'] = df['Fecha pase a producción'].dt.strftime('%d/%m/%Y')
df['Fecha de uso por el personal'] = df['Fecha de uso por el personal'].dt.strftime('%d/%m/%Y')

# Definir el orden final de las columnas
columnas_finales = [
    'Código de proyecto',
    'Área',
    'Prioridad',
    'Presupuesto',
    'Complejidad técnica',
    'Número de dependencias asociadas',
    'Time to market teórico en Q\'s',
    'Fecha de inicio de ejecución',
    'Fecha fin de ejecución',
    'Fecha pase a producción',
    'Fecha de uso por el personal',
    'Time to market' # (Fecha uso personal - Fecha inicio)
]

df_final = df[columnas_finales]

# --- 6. Exportar ---
df_final.to_csv('proyectos_modelo_predictivo.csv', index=False, encoding='utf-8-sig', sep=';')
df_final.to_excel('proyectos_modelo_predictivo.xlsx', index=False)

print("\n--- ¡Simulación completada con éxito! ---")
print(f"Todas las fechas de finalización son iguales o anteriores a 2025-09-30.")
print(f"Archivos guardados: 'proyectos_modelo_predictivo.csv' y 'proyectos_modelo_predictivo.xlsx'")
print("\n--- Vista previa de los primeros 5 proyectos generados ---")
print(df_final.head().to_string())

Generando 4200 proyectos simulados entre 2019-01-01 y 2025-09-30...

--- ¡Simulación completada con éxito! ---
Todas las fechas de finalización son iguales o anteriores a 2025-09-30.
Archivos guardados: 'proyectos_modelo_predictivo.csv' y 'proyectos_modelo_predictivo.xlsx'

--- Vista previa de los primeros 5 proyectos generados ---
  Código de proyecto Área Prioridad  Presupuesto Complejidad técnica  Número de dependencias asociadas  Time to market teórico en Q's Fecha de inicio de ejecución Fecha fin de ejecución Fecha pase a producción Fecha de uso por el personal  Time to market
0       SDATOOL-1639  RIC      Alta       166218               Media                                 2                              4                   01/01/2019             18/10/2019              13/11/2019                   19/11/2019             322
1       SDATOOL-5160  T&C     Media        92827                Alta                                 4                              1                   16/0

In [4]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuración Inicial ---
N_PROYECTOS = 4200
ABSOLUTE_START_DATE = datetime(2019, 1, 1)
ABSOLUTE_END_DATE = datetime(2025, 9, 30) # Límite máximo para fechas REALES

print(f"Generando {N_PROYECTOS} proyectos simulados entre 2019-01-01 y 2025-09-30...")

# --- 1. Generar Códigos Únicos y Variables Categóricas ---

codigos_numericos = np.random.choice(range(1000, 10000), N_PROYECTOS, replace=False)
codigos_proyecto = [f"SDATOOL-{num}" for num in codigos_numericos]

areas = ['CCS', 'ENG', 'RIC', 'RCS', 'FIN', 'T&C', 'GRM', 'CIB']
prioridades = ['Alta', 'Media', 'Baja']
complejidades = ['Alta', 'Media', 'Baja']

data = {
    'Código de proyecto': codigos_proyecto,
    'Área': np.random.choice(areas, N_PROYECTOS, p=[0.1, 0.2, 0.1, 0.1, 0.2, 0.1, 0.1, 0.1]),
    'Prioridad': np.random.choice(prioridades, N_PROYECTOS, p=[0.3, 0.5, 0.2]),
    'Complejidad técnica': np.random.choice(complejidades, N_PROYECTOS, p=[0.3, 0.5, 0.2]),
    'Número de dependencias asociadas': np.random.randint(1, 11, N_PROYECTOS),
}

df = pd.DataFrame(data)

# --- 2. Generar Presupuesto (Correlacionado con Complejidad) ---

def calcular_presupuesto(row):
    base_presupuesto = 50000
    if row['Complejidad técnica'] == 'Alta': factor_complejidad = 2.5
    elif row['Complejidad técnica'] == 'Media': factor_complejidad = 1.5
    else: factor_complejidad = 1.0
    factor_dependencias = 1 + (row['Número de dependencias asociadas'] / 10)
    ruido = np.random.randint(-15000, 15000)
    # El TTM Teórico ya no existe, usamos un factor base
    presupuesto = (base_presupuesto * factor_complejidad * factor_dependencias * 1.5) + ruido
    return max(10000, int(presupuesto))

df['Presupuesto'] = df.apply(calcular_presupuesto, axis=1)

# --- 3. Simulación de TTM Real y Retraso ---
# Esta es la "verdad" que el modelo intentará aprender.

def simular_ttm_y_retraso(row):
    # 3.1. Simular TTM Base (Planificado "Ideal")
    # Los proyectos complejos se planifican para durar más
    if row['Complejidad técnica'] == 'Alta':
        ttm_planificado_base_dias = np.random.randint(270, 365)
    elif row['Complejidad técnica'] == 'Media':
        ttm_planificado_base_dias = np.random.randint(150, 270)
    else:
        ttm_planificado_base_dias = np.random.randint(60, 150)

    # 3.2. Simular el Retraso (El "error" que crea los outliers)
    # Esta es la señal que tu modelo buscará
    retraso_dias = 0
    
    # Proyectos complejos y con dependencias se retrasan más
    if row['Complejidad técnica'] == 'Alta' or row['Número de dependencias asociadas'] > 7:
        # Tienen alta probabilidad de un retraso atípico
        retraso_dias = np.random.choice(
            [np.random.randint(0, 30), np.random.randint(30, 121), np.random.randint(-30, 0)],
            p=[0.4, 0.5, 0.1] # 50% de chance de ser atípico, 10% de terminar antes
        )
    else:
        # Proyectos normales, variación +/- 30 días
        retraso_dias = np.random.randint(-30, 31)
        
    # 3.3. Calcular TTM Real
    ttm_real_dias = ttm_planificado_base_dias + retraso_dias
    
    return pd.Series([ttm_planificado_base_dias, ttm_real_dias, retraso_dias])

# Aplicar la simulación
df[['ttm_planificado_dias', 'ttm_real_dias', 'retraso_simulado_dias']] = df.apply(simular_ttm_y_retraso, axis=1)

# Asegurar que el TTM real sea al menos 30 días
df['ttm_real_dias'] = df['ttm_real_dias'].clip(lower=30) 

# --- 4. Calcular Fechas (Lógica Invertida para cumplir 2025-09-30) ---

# 4.1. Generar 'Fecha de uso por el personal' (REAL) PRIMERO
MIN_END_DATE = ABSOLUTE_START_DATE + timedelta(days=90) # Proy. dura al menos 90 días
DIAS_RANGO_FINALIZACION = (ABSOLUTE_END_DATE - MIN_END_DATE).days
dias_random_final = np.random.randint(0, DIAS_RANGO_FINALIZACION, N_PROYECTOS)
df['Fecha de uso por el personal'] = MIN_END_DATE + pd.to_timedelta(dias_random_final, unit='d')

# 4.2. Calcular 'Fecha de inicio' HACIA ATRÁS, usando el TTM REAL
df['Fecha de inicio de ejecución'] = df['Fecha de uso por el personal'] - pd.to_timedelta(df['ttm_real_dias'], unit='d')

# 4.3. Anclar las fechas de inicio que sean < 2019-01-01
df['Fecha de inicio de ejecución'] = df['Fecha de inicio de ejecución'].clip(lower=ABSOLUTE_START_DATE)

# 4.4. Calcular la 'Fecha planificada' usando el TTM Planificado
# Fecha Planificada = Fecha Inicio + TTM Planificado
df['Fecha de uso por el personal planificada'] = df['Fecha de inicio de ejecución'] + pd.to_timedelta(df['ttm_planificado_dias'], unit='d')

# 4.5. Generar fechas intermedias
df['Fecha pase a producción'] = df['Fecha de uso por el personal'] - pd.to_timedelta(np.random.randint(1, 11, N_PROYECTOS), unit='d')
df['Fecha fin de ejecución'] = df['Fecha pase a producción'] - pd.to_timedelta(np.random.randint(7, 30, N_PROYECTOS), unit='d')

# 4.6. Asegurarse que las fechas intermedias no sean anteriores al inicio
df['Fecha pase a producción'] = df['Fecha pase a producción'].clip(lower=df['Fecha de inicio de ejecución'])
df['Fecha fin de ejecución'] = df['Fecha fin de ejecución'].clip(lower=df['Fecha de inicio de ejecución'])

# --- 5. Calcular TTM final y Limpieza ---

# RE-CALCULAR el 'Time to market' (REAL) basado en las fechas finales (que podrían estar "ancladas")
delta_ttm = (df['Fecha de uso por el personal'] - df['Fecha de inicio de ejecución'])
df['Time to market'] = delta_ttm.dt.days

# Formatear fechas a XX/XX/XXXX
date_cols = ['Fecha de inicio de ejecución', 'Fecha fin de ejecución', 
             'Fecha pase a producción', 'Fecha de uso por el personal', 
             'Fecha de uso por el personal planificada']
for col in date_cols:
    df[col] = df[col].dt.strftime('%d/%m/%Y')

# Definir el orden final de las columnas
columnas_finales = [
    'Código de proyecto',
    'Área',
    'Prioridad',
    'Presupuesto',
    'Complejidad técnica',
    'Número de dependencias asociadas',
    'Fecha de inicio de ejecución',
    'Fecha de uso por el personal planificada', # <-- Nueva columna
    'Fecha fin de ejecución',
    'Fecha pase a producción',
    'Fecha de uso por el personal', # Esta es la fecha REAL
    'Time to market' # Este es el TTM REAL
]

df_final = df[columnas_finales]

# --- 6. Exportar ---
df_final.to_csv('proyectos_modelo_predictivo_v2.csv', index=False, encoding='utf-8-sig', sep=';')
df_final.to_excel('proyectos_modelo_predictivo_v2.xlsx', index=False)

print("\n--- ¡Simulación completada con éxito! ---")
print(f"Se eliminó 'TTM Teórico' y se añadió 'Fecha de uso por el personal planificada'.")
print(f"Archivos guardados: 'proyectos_modelo_predictivo_v2.csv' y 'proyectos_modelo_predictivo_v2.xlsx'")
print("\n--- Vista previa de los primeros 5 proyectos generados ---")
print(df_final.head().to_string())

print("\n--- Verificación de Retrasos (Planificado vs Real) ---")
# Para verificar, volvemos a cargar las fechas para comparar
df_check = pd.DataFrame()
df_check['Planificada'] = pd.to_datetime(df_final['Fecha de uso por el personal planificada'], format='%d/%m/%Y')
df_check['Real'] = pd.to_datetime(df_final['Fecha de uso por el personal'], format='%d/%m/%Y')
df_check['Diferencia (días)'] = (df_check['Real'] - df_check['Planificada']).dt.days
print(df_check.head())
print("\nDescripción de la Diferencia (Positivo = Retraso, Negativo = Adelanto):")
print(df_check['Diferencia (días)'].describe())

Generando 4200 proyectos simulados entre 2019-01-01 y 2025-09-30...

--- ¡Simulación completada con éxito! ---
Se eliminó 'TTM Teórico' y se añadió 'Fecha de uso por el personal planificada'.
Archivos guardados: 'proyectos_modelo_predictivo_v2.csv' y 'proyectos_modelo_predictivo_v2.xlsx'

--- Vista previa de los primeros 5 proyectos generados ---
  Código de proyecto Área Prioridad  Presupuesto Complejidad técnica  Número de dependencias asociadas Fecha de inicio de ejecución Fecha de uso por el personal planificada Fecha fin de ejecución Fecha pase a producción Fecha de uso por el personal  Time to market
0       SDATOOL-9003  CCS      Alta        97666                Baja                                 2                   14/11/2020                               18/01/2021             12/12/2020              03/01/2021                   07/01/2021              54
1       SDATOOL-5237  RIC     Media       358724                Alta                                 9                   